# Lab 03: Probability and Maximum Likelihood

## Gaussian Distribution

One of the most important probability density functions is the Gaussian distribution, which is defined as follows:

$$
  p(x) = \mathcal{N}(x \mid \mu, \sigma) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(x - \mu)^2}{2\sigma^2}\right) .
$$

The variables $\mu \in \mathbb{R}$ and $\sigma \in \mathbb{R}_{> 0}$ are the mean and standard deviation of the distribution, respectively. 

In [ ]:
import numpy as np

def gaussian(x: np.ndarray, mu: np.ndarray, sigma: np.ndarray) -> np.ndarray:
  """Calculate the Gaussian probability density function.

  Args:
    x: The input data points, shape (N,).
    mu: The mean of the Gaussian distribution for each data point, shape (N,).
    sigma: The standard deviation of the Gaussian distribution, shape (N,).

  Returns:
    The probability density, shape (N,).
  """
  return (1 / np.sqrt(2 * np.pi * sigma**2)) * np.exp(-(x - mu)**2 / (2 * sigma**2))

NumPy provides a built-in function for the Gaussian distribution, such as `numpy.random.normal` for sampling and `scipy.stats.norm.pdf` for the probability density function. However, we implemented our own version to understand the underlying math.

The **parameters** of the distribution, and they control the shape of the probability density function.

Note: Run `%pip install matplotlib` and `%pip install scipy` first if you don't have it installed.
Other Note: I had to do `%pip install -U ipywidgets==7.7.1` to get it to work in vscode

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("bmh")
plt.rcParams["font.family"] = ["Helvetica"]

from ipywidgets import interact, FloatSlider


def plot_gaussian(mu, sigma):
  x = np.linspace(-10, 10, 300)
  p = gaussian(x, mu, sigma)

  plt.figure(figsize=(7, 3))
  plt.plot(x, p)
  plt.xlim(-10, 10)
  plt.ylim(0, 1)
  plt.xlabel(r"$x$"); plt.ylabel(r"$\mathcal{N}(x \mid \mu, \sigma^2)$")
  plt.show()

interact(
  plot_gaussian,
  mu=FloatSlider(value=0, min=-5, max=5, step=0.1),
  sigma=FloatSlider(value=1, min=0.1, max=10, step=0.1)
)

## Likelihood Functions

Let's say we have a data point $x_n$, and we **assume** that it comes from a Gaussian with a particular value for $\mu$ and a constant value for $\sigma^2 = 1$.
The likelihood of $\mu$ given $x_n$ is the probability density of $x_n$ under that Gaussian $\mathcal{N}(x_n \mid \mu, 1)$.

Note: Run `%pip install scipy` first if you don't have it installed.


In [ ]:
import scipy.stats as st


def likelihood(mu, x_n):
  # In numpy etc, mu is usually called "loc" and sigma is called "scale"
  return st.norm.pdf(x_n, loc=mu, scale=1)

Given multiple data points $\mathcal{D} = \{x_1, \ldots, x_N\}$ **i.i.d.** from a Gaussian distribution, the likelihood of $\mu$ given $\mathcal{D}$ is the product of the likelihoods for each data point:

$$
  p(\mathcal{D} \mid \mu) = \prod_{n=1}^N \mathcal{N}(x_n \mid \mu, 1) .
$$

Equivalently, we can work with the log-likelihood, which is the sum of the log-likelihoods for each data point:

$$
  \log p(\mathcal{D} \mid \mu) = \sum_{n=1}^N \log \mathcal{N}(x_n \mid \mu, 1) .
$$


In [ ]:
def log_likelihood(mu, data):
  return np.sum(st.norm.logpdf(data, loc=mu, scale=1))

In practice, we do not know for sure whether the data points are actually i.i.d. from a Gaussian distribution.
We can still use a Gaussian likelihoods as an assumption.

## Maximum Likelihood Estimation

Let's say we have the following data points, obtained by measuring the heights of 10 randomly selected students:

In [ ]:
data = np.array(
    [165.2, 178.4, 182.1, 159.7, 171.3, 188.6, 163.9, 175.0, 169.4, 183.2], 
)

Remember that when defining the log-likelihood function above, we did not specify the value of $\mu$---we left it as a variable. 
The question is thus: Given the data above, what is the "best" value of $\mu$ that explains the data?

**Maximum likelihood estimation (MLE)** is a method for finding the parameter value of (here, $\mu$) that maximizes the likelihood (or equivalently, the log-likelihood) of the data.


In [ ]:
mus = np.linspace(150, 190, 300)
logliks = [log_likelihood(mu, data) for mu in mus]
mle = np.argmax(logliks)

plt.plot(mus, logliks, label="Log-Likelihood")
plt.vlines(mus[mle], np.min(logliks), np.max(logliks), color="red", linestyle="--", label="MLE")

plt.title(r"Log-Likelihood of $\mathcal{D}$ given different $\mu$")
plt.xlabel(r"$\mu$")
plt.ylabel(r"$\log p(\mathcal{D} \mid \mu)$")
plt.xlim(150, 190)
plt.ylim(np.min(logliks))

plt.legend()

## Linear Regression as MLE

In linear regression, we have a dataset $\mathcal{D} = \{(\mathbf{x}_1, y_1), \ldots, (\mathbf{x}_N, y_N)\}$ and we want to find the parameter $\boldsymbol{\theta}$ that best explains the data. We model the output as:

$$
  p(y \mid \mathbf{x}, \boldsymbol{\theta}) = \mathcal{N}(y \mid f(\mathbf{x}; \boldsymbol{\theta}), 1), \qquad f(\mathbf{x}; \boldsymbol{\theta}) = \boldsymbol{\theta}^\top \mathbf{x} .
$$

As shown in lecture, maximizing the log-likelihood is equivalent to minimizing the squared error:

$$
  \boldsymbol{\theta}^* = \arg\max_{\boldsymbol{\theta}} \log p(\mathcal{D} \mid \boldsymbol{\theta}) = \arg\min_{\boldsymbol{\theta}} \sum_{n=1}^N (y_n - f(\mathbf{x}_n; \boldsymbol{\theta}))^2 .
$$

Let's verify this with a simple 1D example. We generate data from a known linear function with Gaussian noise:

In [ ]:
np.random.seed(42)
X = np.linspace(0, 10, 30)
TRUE_THETA = 2.5
y = TRUE_THETA * X + np.random.normal(0, 1, size=len(X))

Now we define the log-likelihood and squared error as functions of $\theta$:

In [ ]:
def linreg_log_likelihood(theta, X, y):
    y_pred = theta * X
    return np.sum(st.norm.logpdf(y, loc=y_pred, scale=1))


def squared_error(theta, X, y):
    y_pred = theta * X
    return 1/2 * np.sum((y - y_pred)**2)

We can now plot both objectives over a range of $\theta$ values and confirm they peak/bottom at the same point:

In [ ]:
thetas = np.linspace(1, 4, 300)
logliks = [linreg_log_likelihood(t, X, y) for t in thetas]
sq_errs = [squared_error(t, X, y) for t in thetas]

theta_mle = thetas[np.argmax(logliks)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(thetas, logliks)
axes[0].axvline(theta_mle, color="red", linestyle="--", label=f"MLE = {theta_mle:.2f}")
axes[0].set_title(r"Log-Likelihood $\log p(\mathcal{D} \mid \theta)$")
axes[0].set_xlabel(r"$\theta$")
axes[0].legend()

axes[1].plot(thetas, sq_errs)
axes[1].axvline(theta_mle, color="red", linestyle="--", label=f"Minimum = {theta_mle:.2f}")
axes[1].set_title(r"Squared Error $\sum_n (y_n - \theta x_n)^2$")
axes[1].set_xlabel(r"$\theta$")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"MLE estimate: {theta_mle:.3f} (true value: {TRUE_THETA})")

The red line lands at the same $\theta$ in both plots.